# RadLE Medical Workbench Runtime

Experimental runner for `medgemma_1_5_4b`, `llava_med_mistral_7b`, `internvl3_5_8b`, and `octomed_7b`. Run one selected model at a time on a GPU-backed Colab Enterprise or Vertex Workbench runtime.

Run the cells top to bottom. This is the Workbench-specific fork; the original Colab notebook stays at `RadLE_Medical_Custom_Runtime.ipynb`. The runtime setup cell finds a writable runtime root, pulls the latest repo code, and prints the commit being used.

Storage rules:
- Workbench and Colab Enterprise should not rely on `google.colab.drive.mount()`.
- Use `DATASET_ROOT_OVERRIDE`, `RADLE_DATASET_ROOT`, or `RADLE_DATASET_GCS_URI` so the runtime can see a folder that contains `RadLE v2 Master Data`.


# Data Staging Contract

The RadLE v2 image corpus is treated as a frozen snapshot for Workbench runs. Google Drive remains the human/source archive, but inference should read from local Workbench disk after staging from a private regional GCS bucket in the same region as the Workbench VM.

Expected cloud layout:
- Dataset snapshot: `gs://radle-medical-data-toronto/datasets/radle-v2-frozen-2026-06-29/RadLE v2 Master Data/`
- Results archive: `gs://radle-medical-data-toronto/runs/`

Execution data flow:
1. One-time staging: Standard Colab mounts Drive and syncs `RadLE v2 Master Data` to the frozen GCS snapshot.
2. Workbench setup: this notebook copies the frozen snapshot from GCS to local disk through `RADLE_DATASET_GCS_URI`.
3. Benchmark execution: vLLM/SGLang reads images from the local `RadLE v2 Master Data` folder.
4. Results preservation: after export, the run folder is synced back to GCS through `RADLE_RESULTS_GCS_URI`.

Do not run inference directly from Google Drive or a Drive FUSE mount.


In [ ]:
# ==========================================
# 0. DATA STAGING CONFIG
# ==========================================
import os
import pathlib

# Default bucket created for the Toronto Workbench pipeline. Override these
# before running the cell if the bucket or snapshot name changes.
GCS_BUCKET = os.environ.get("RADLE_GCS_BUCKET", "radle-medical-data-toronto")
DATASET_SNAPSHOT_ID = os.environ.get(
    "RADLE_DATASET_SNAPSHOT_ID",
    "radle-v2-frozen-2026-06-29",
)
DATASET_GCS_PREFIX = f"gs://{GCS_BUCKET}/datasets/{DATASET_SNAPSHOT_ID}"
DATASET_GCS_URI = f"{DATASET_GCS_PREFIX}/RadLE v2 Master Data"

os.environ.setdefault("RADLE_DATASET_GCS_URI", DATASET_GCS_URI)
os.environ.setdefault("RADLE_RESULTS_GCS_URI", f"gs://{GCS_BUCKET}/runs")

default_workbench_dataset_root = pathlib.Path.home() / "radle_dataset" / "RadLE v2 Dataset"
models_ssd = pathlib.Path("/mnt/disks/models_ssd")
if models_ssd.exists() and os.path.ismount(str(models_ssd)):
    os.environ.setdefault(
        "RADLE_LOCAL_DATASET_ROOT",
        str(models_ssd / "radle_dataset" / "RadLE v2 Dataset"),
    )
    os.environ.setdefault(
        "RADLE_RUNTIME_CACHE_ROOT",
        str(models_ssd / "radle_runtime_cache"),
    )
else:
    os.environ.setdefault("RADLE_LOCAL_DATASET_ROOT", str(default_workbench_dataset_root))
    print("/mnt/disks/models_ssd is not mounted; using home-disk Workbench fallbacks.")

for key in [
    "RADLE_DATASET_GCS_URI",
    "RADLE_LOCAL_DATASET_ROOT",
    "RADLE_RUNTIME_CACHE_ROOT",
    "RADLE_RESULTS_GCS_URI",
]:
    print(f"{key}=", os.environ.get(key, "<not set>"))


# ==========================================
# 0.5. CACHE DATASET TO LOCAL SSD
# ==========================================
Copy the frozen snapshot from GCS directly to the local NVMe SSD to ensure the L4 GPUs are not bottlenecked by FUSE network latency.

In [ ]:
import os
import pathlib
import subprocess

gcs_uri = os.environ.get("RADLE_DATASET_GCS_URI")
dataset_root = pathlib.Path(
    os.environ.get(
        "RADLE_LOCAL_DATASET_ROOT",
        str(pathlib.Path.home() / "radle_dataset" / "RadLE v2 Dataset"),
    )
).expanduser()
master_data_dir = dataset_root / "RadLE v2 Master Data"

if not gcs_uri:
    raise RuntimeError("RADLE_DATASET_GCS_URI is not set. Run the data staging config cell first.")

source = gcs_uri.rstrip("/")
if not source.endswith("/RadLE v2 Master Data"):
    source = f"{source}/RadLE v2 Master Data"

if master_data_dir.exists() and any(master_data_dir.iterdir()):
    print("Dataset already cached:", master_data_dir)
else:
    dataset_root.mkdir(parents=True, exist_ok=True)
    print(f"Caching {source} to {dataset_root}...")
    subprocess.run(["gcloud", "storage", "cp", "-r", source, str(dataset_root)], check=True)
    print("Dataset cached successfully.")

if not master_data_dir.exists():
    raise RuntimeError(f"Expected dataset folder was not created: {master_data_dir}")

os.environ["RADLE_LOCAL_DATASET_ROOT"] = str(dataset_root)
os.environ["DATASET_ROOT_OVERRIDE"] = str(dataset_root)
print("Dataset root:", dataset_root)
print("Master data:", master_data_dir)

In [ ]:
# ==========================================
# 1. RUNTIME SETUP: REPO CODE ONLY
# ==========================================
import os
import pathlib
import subprocess
import sys


def path_is_writable(path):
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".radle_write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink(missing_ok=True)
        return True
    except Exception:
        return False


def select_runtime_root():
    override = os.environ.get("RADLE_RUNTIME_ROOT")
    if override:
        root = pathlib.Path(override).expanduser()
        root.mkdir(parents=True, exist_ok=True)
        return root

    for candidate in (pathlib.Path("/content"), pathlib.Path.home(), pathlib.Path.cwd()):
        if candidate.exists() and path_is_writable(candidate):
            return candidate

    raise RuntimeError("Set RADLE_RUNTIME_ROOT to a writable runtime path.")


def find_existing_repo_dir():
    override = os.environ.get("RADLE_REPO_DIR")
    if override:
        return pathlib.Path(override).expanduser()

    cwd = pathlib.Path.cwd()
    for candidate in (cwd, cwd.parent):
        if (candidate / "src" / "radle_medical_custom_runtime.py").exists():
            return candidate

    return RUNTIME_ROOT / "RadLE_CRASH_Lab"


TOKEN_FILE_BY_SECRET = {
    "GITHUB_TOKEN": ".github_token",
    "HF_TOKEN": ".hf_token",
    "HUGGING_FACE_HUB_TOKEN": ".hf_token",
}


def get_secret(name, *fallback_names):
    for env_name in (name, *fallback_names):
        value = os.environ.get(env_name)
        if value:
            return value

    for secret_name in (name, *fallback_names):
        token_file_name = TOKEN_FILE_BY_SECRET.get(secret_name)
        if not token_file_name:
            continue
        token_file = pathlib.Path.home() / token_file_name
        if token_file.exists():
            value = token_file.read_text(encoding="utf-8").strip()
            if value:
                return value

    try:
        from google.colab import userdata
    except Exception:
        return None

    for secret_name in (name, *fallback_names):
        try:
            value = userdata.get(secret_name)
        except Exception:
            value = None
        if value:
            return value
    return None


RUNTIME_ROOT = select_runtime_root()
REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_DIR = find_existing_repo_dir()
SRC_DIR = REPO_DIR / "src"
MODULE_PATH = SRC_DIR / "radle_medical_custom_runtime.py"
github_token = get_secret("GITHUB_TOKEN")
if github_token:
    os.environ["GITHUB_TOKEN"] = github_token
    print("GitHub token loaded from environment, Colab Secrets, or ~/.github_token; value hidden.")
else:
    print("No GitHub token loaded. Existing local checkouts may still work, but cloning/pulling this private repo needs GITHUB_TOKEN or ~/.github_token.")


def authenticated_repo_url():
    if not github_token:
        return REPO_URL
    return REPO_URL.replace("https://", f"https://x-access-token:{github_token}@")


def mask_secret(value):
    text = str(value)
    if github_token:
        text = text.replace(github_token, "***")
    return text


def run_command(args, check=True, env=None):
    result = subprocess.run(args, text=True, capture_output=True, env=env)
    stdout = mask_secret(result.stdout)
    stderr = mask_secret(result.stderr)
    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)
    if check and result.returncode != 0:
        safe_args = [mask_secret(arg) for arg in args[:3]]
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {safe_args}")
    return result


print("Runtime root:", RUNTIME_ROOT)
print("Repo dir:", REPO_DIR)

repo_url = authenticated_repo_url()
if (REPO_DIR / ".git").exists():
    if github_token:
        run_command(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", repo_url])
    pull_result = run_command(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    if pull_result.returncode != 0:
        raise RuntimeError(
            "git pull failed; the runtime checkout may be stale or dirty. Restart the runtime "
            f"or remove {REPO_DIR}, then rerun this setup cell."
        )
elif not MODULE_PATH.exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout and {MODULE_PATH} was not found. "
            "Set RADLE_REPO_DIR to a clean checkout, remove that folder, or restart the runtime."
        )

    if not github_token:
        raise RuntimeError(
            "This private GitHub repo needs GITHUB_TOKEN as an environment variable, "
            "Colab secret, or ~/.github_token so the runtime can clone the medical runtime module."
        )

    run_command(["git", "clone", repo_url, str(REPO_DIR)])

if not MODULE_PATH.exists():
    raise RuntimeError(f"Medical runtime module not found after setup: {MODULE_PATH}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

commit_result = run_command(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    check=False,
)
if commit_result.returncode == 0:
    print("Using repo commit:", commit_result.stdout.strip())


In [ ]:
# ==========================================
# 2. PYTHON DEPENDENCIES + CACHE ROUTING
# ==========================================
# Set INSTALL_SERVER_PACKAGES=False if your custom runtime image already has vLLM/SGLang.
INSTALL_SERVER_PACKAGES = True
SERVER_ENGINE = "vllm"  # "vllm" or "sglang"

# The plain vLLM wheel can resolve to a CUDA-13 build on Colab and fail with
# ImportError: libcudart.so.13. Use the explicit CUDA 12.9 wheel for NVIDIA Colab.
VLLM_VERSION = "0.23.0"
VLLM_CUDA_VARIANT = "cu129"
VLLM_WHEEL_URL = (
    f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VERSION}/"
    f"vllm-{VLLM_VERSION}%2B{VLLM_CUDA_VARIANT}-cp38-abi3-manylinux_2_28_x86_64.whl"
)
TORCH_EXTRA_INDEX_URL = f"https://download.pytorch.org/whl/{VLLM_CUDA_VARIANT}"

# Workbench runtimes do not always have /content. Keep /content as the Colab
# default, but let RADLE_RUNTIME_ROOT or RADLE_RUNTIME_CACHE_ROOT redirect caches.
ssd_mount = pathlib.Path("/mnt/disks/models_ssd")
if ssd_mount.exists() and os.path.ismount(str(ssd_mount)):
    default_cache_root = ssd_mount
else:
    default_cache_root = RUNTIME_ROOT / "radle_runtime_cache"

RUNTIME_CACHE_ROOT = pathlib.Path(
    os.environ.get("RADLE_RUNTIME_CACHE_ROOT", str(default_cache_root))
).expanduser()
cache_paths = {
    "HF_HOME": RUNTIME_CACHE_ROOT / "huggingface",
    "HUGGINGFACE_HUB_CACHE": RUNTIME_CACHE_ROOT / "huggingface" / "hub",
    "TRANSFORMERS_CACHE": RUNTIME_CACHE_ROOT / "huggingface" / "transformers",
    "PIP_CACHE_DIR": RUNTIME_CACHE_ROOT / "pip",
    "VLLM_CACHE_ROOT": RUNTIME_CACHE_ROOT / "vllm",
    "XDG_CACHE_HOME": RUNTIME_CACHE_ROOT / "xdg",
}
for key, path in cache_paths.items():
    path.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(path)

print("Cache root:", RUNTIME_CACHE_ROOT)
df_targets = ["/", str(RUNTIME_ROOT)]
if pathlib.Path("/content").exists():
    df_targets.append("/content")
df_targets = list(dict.fromkeys(df_targets))
run_command(["df", "-h", *df_targets], check=False)
run_command(["nvidia-smi"], check=False)

base_packages = [
    "openai",
    "pandas",
    "google-cloud-aiplatform",
    "huggingface_hub",
]
# Do not force --upgrade here. Colab custom runtimes ship pinned packages.
# The medical side path uses local OpenAI-compatible serving; native Anthropic
# and Gemini SDK imports are stubbed by radle_medical_custom_runtime when absent.
run_command([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--cache-dir",
    os.environ["PIP_CACHE_DIR"],
    *base_packages,
])

# Workbench images may ship a scikit-learn wheel built against a different
# NumPy ABI. Transformers treats sklearn as optional, but if the broken wheel
# is present, vLLM's transformers import can fail before model loading.
# RadLE/vLLM do not need sklearn for this path, so remove it from the runtime.
run_command([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "scikit-learn",
    "sklearn",
], check=False)

if INSTALL_SERVER_PACKAGES:
    if SERVER_ENGINE == "vllm":
        # Remove any preexisting CUDA-13 vLLM wheel before installing the CUDA-12.9 build.
        run_command([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"], check=False)
        run_command([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--cache-dir",
            os.environ["PIP_CACHE_DIR"],
            VLLM_WHEEL_URL,
            "--extra-index-url",
            TORCH_EXTRA_INDEX_URL,
        ])
        run_command([
            sys.executable,
            "-c",
            (
                "from vllm.platforms import current_platform; "
                "import vllm; "
                "print('vLLM import OK:', vllm.__version__, vllm.__file__, current_platform)"
            ),
        ])
    elif SERVER_ENGINE == "sglang":
        run_command([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--cache-dir",
            os.environ["PIP_CACHE_DIR"],
            "sglang[all]",
        ])
    else:
        raise ValueError("SERVER_ENGINE must be 'vllm' or 'sglang'.")

print("Dependency install finished. Continue unless a later import or server cell raises an error.")


# ==========================================
# 2.5. OPTIONAL VERTEX AI EXPERIMENTS SETUP
# ==========================================
Vertex AI Experiments logging is disabled by default to avoid creating extra cloud resources during smoke/debug runs.

In [ ]:
ENABLE_VERTEX_EXPERIMENTS = os.environ.get("RADLE_ENABLE_VERTEX_EXPERIMENTS", "0") == "1"

if ENABLE_VERTEX_EXPERIMENTS:
    import google.cloud.aiplatform as aiplatform

    aiplatform.init(
        project="crashlab-synthetic",
        location="northamerica-northeast2",
        experiment="radle-medical-benchmark",
    )
    print("Vertex AI Experiments initialized.")
else:
    print("Vertex AI Experiments disabled. Set RADLE_ENABLE_VERTEX_EXPERIMENTS=1 before this cell to enable cloud logging.")

In [ ]:
# ==========================================
# 3. IMPORT MEDICAL RUNTIME HELPERS
# ==========================================
import importlib

# Import the medical helper first. It installs lightweight stubs for native
# provider SDKs that this local OpenAI-compatible medical path does not use.
import radle_medical_custom_runtime as medical_runtime

medical_runtime = importlib.reload(medical_runtime)
radle_benchmark = medical_runtime.radle_benchmark
radle_benchmark = importlib.reload(radle_benchmark)
medical_runtime.radle_benchmark = radle_benchmark
medical_runtime.configure_cache_environment(str(RUNTIME_CACHE_ROOT))

print("Benchmark module:", radle_benchmark.__file__)
print("Medical runtime module:", medical_runtime.__file__)
print("Supported medical models:")
medical_runtime.print_model_roster()
medical_runtime.print_runtime_resources()


In [ ]:
# ==========================================
# 4. MODEL, SERVER, AND RUN CONFIG
# ==========================================
import os
from pathlib import Path

# Full-run defaults after a clean 20-case MedGemma smoke.
# For another shakedown, set TEST_LIMIT to a small integer and change RUN_LABEL.
# When switching models, stop the server or restart the runtime and use a new RUN_LABEL.
SELECTED_MODEL_NAME = "medgemma_1_5_4b"
# SELECTED_MODEL_NAME = "llava_med_mistral_7b"
# SELECTED_MODEL_NAME = "internvl3_5_8b"
# SELECTED_MODEL_NAME = "octomed_7b"

TEST_LIMIT = None
RUN_LABEL = "medical_full_263_cases"
RESUME = True
MAX_OUTPUT_TOKENS = 2048
EXPECTED_FULL_CASES = 263

IS_STANDARD_COLAB = medical_runtime.is_standard_colab()
DEFAULT_DRIVE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset"
)
WORKBENCH_DATASET_ROOT = Path(
    os.environ.get(
        "RADLE_LOCAL_DATASET_ROOT",
        str(Path.home() / "radle_dataset" / "RadLE v2 Dataset"),
    )
).expanduser()

DATASET_ROOT_OVERRIDE = os.environ.get("DATASET_ROOT_OVERRIDE") or os.environ.get("RADLE_DATASET_ROOT", "")
RADLE_DATASET_GCS_URI = os.environ.get(
    "RADLE_DATASET_GCS_URI",
    "gs://radle-medical-data-toronto/datasets/radle-v2-frozen-2026-06-29/RadLE v2 Master Data",
)
os.environ["RADLE_DATASET_GCS_URI"] = RADLE_DATASET_GCS_URI


def has_master_data(path):
    return (Path(path) / "RadLE v2 Master Data").exists()


def maybe_mount_standard_colab_drive():
    if DEFAULT_DRIVE_DATASET_ROOT.exists():
        return
    try:
        from google.colab import drive
    except Exception:
        return
    print("Mounting Google Drive for the default RadLE dataset path...")
    drive.mount("/content/drive")


def copy_dataset_from_gcs(dataset_root):
    dataset_root = Path(dataset_root).expanduser()
    if has_master_data(dataset_root):
        print("Dataset already present:", dataset_root)
        return

    source = RADLE_DATASET_GCS_URI.rstrip("/")
    if not source.endswith("/RadLE v2 Master Data"):
        source = f"{source}/RadLE v2 Master Data"

    dataset_root.mkdir(parents=True, exist_ok=True)
    print("Copying RadLE master data from private GCS into:", dataset_root)
    run_command(["gcloud", "storage", "cp", "-r", source, str(dataset_root)])

    if not has_master_data(dataset_root):
        raise RuntimeError(f"GCS copy finished but RadLE v2 Master Data was not found under {dataset_root}")


if (not IS_STANDARD_COLAB) and DATASET_ROOT_OVERRIDE.startswith("/content/drive"):
    raise RuntimeError(
        "DATASET_ROOT_OVERRIDE points to /content/drive, but Workbench/Colab Enterprise "
        "should not rely on google.colab.drive.mount(). Use a local copied folder or GCS."
    )

if IS_STANDARD_COLAB:
    maybe_mount_standard_colab_drive()
else:
    copy_dataset_from_gcs(WORKBENCH_DATASET_ROOT)

_dataset_candidates = []
if DATASET_ROOT_OVERRIDE:
    _dataset_candidates.append(Path(DATASET_ROOT_OVERRIDE).expanduser())
if IS_STANDARD_COLAB:
    _dataset_candidates.append(DEFAULT_DRIVE_DATASET_ROOT)
else:
    _dataset_candidates.append(WORKBENCH_DATASET_ROOT)

seen_candidates = set()
for _candidate in _dataset_candidates:
    candidate_key = str(_candidate)
    if candidate_key in seen_candidates:
        continue
    seen_candidates.add(candidate_key)
    if has_master_data(_candidate):
        dataset_root = _candidate
        break
else:
    checked = "\n".join(f"- {candidate}" for candidate in _dataset_candidates)
    raise RuntimeError(
        "Could not find the RadLE dataset root. Expected one candidate to contain "
        "'RadLE v2 Master Data'.\n"
        f"RADLE_DATASET_GCS_URI={RADLE_DATASET_GCS_URI}\n"
        f"Checked:\n{checked}"
    )

run_paths = medical_runtime.build_medical_run_paths(
    dataset_root,
    model_name=SELECTED_MODEL_NAME,
    run_label=RUN_LABEL,
)

master_images_folder = run_paths["master_images_folder"]
raw_results_csv = run_paths["raw_results_csv"]
raw_backup_dir = run_paths["raw_backup_dir"]
scorer_csv = run_paths["scorer_view_csv"]

image_index_for_expected_cases = radle_benchmark.build_image_index(master_images_folder)
all_case_ids = sorted(image_index_for_expected_cases.keys(), key=radle_benchmark.numeric_case_sort_key)
expected_case_ids = all_case_ids if TEST_LIMIT is None else all_case_ids[:TEST_LIMIT]
expected_case_count = len(expected_case_ids)

if not expected_case_ids:
    raise RuntimeError(f"No benchmark cases found under {master_images_folder}")
if TEST_LIMIT is None and expected_case_count != EXPECTED_FULL_CASES:
    raise RuntimeError(
        f"Full run expected {EXPECTED_FULL_CASES} cases, but found {expected_case_count}. "
        "Check dataset staging before launching the full run."
    )

model_runtime = medical_runtime.get_model(SELECTED_MODEL_NAME)
active_model = [model_runtime.benchmark_config()]

# Local server config. Use START_SERVER=False if you already started an OpenAI-compatible endpoint.
START_SERVER = True
HOST = "127.0.0.1"
PORT = 8000
BASE_URL = f"http://{HOST}:{PORT}/v1"
TENSOR_PARALLEL_SIZE = 1
GPU_MEMORY_UTILIZATION = model_runtime.default_gpu_memory_utilization
MAX_MODEL_LEN = 4096
SERVER_TIMEOUT_SECONDS = 900
# Gemma 3 / MedGemma rejects float16 in vLLM because of numerical instability.
# L4 GPUs support bfloat16, so use it for MedGemma and keep float16 for the other roster models.
MODEL_DTYPE = "bfloat16" if SELECTED_MODEL_NAME == "medgemma_1_5_4b" else "float16"
EXTRA_SERVER_ARGS = ["--dtype", MODEL_DTYPE]

repair_output_csv = run_paths["repair_results_csv"]
repair_call_log_csv = run_paths["repair_call_log_csv"]
repair_plan_csv = run_paths["repair_plan_csv"]
repair_backup_dir = run_paths["repair_backup_dir"]
final_results_csv = run_paths["final_results_csv"]
final_manifest_json = run_paths["final_manifest_json"]
public_release_dir = run_paths["public_release_dir"]

print("Standard Colab:", IS_STANDARD_COLAB)
print("Selected model:", SELECTED_MODEL_NAME)
print("Run label:", RUN_LABEL)
print("Test limit:", TEST_LIMIT if TEST_LIMIT is not None else "full")
print("Expected cases this run:", expected_case_count)
print("Dataset GCS URI:", RADLE_DATASET_GCS_URI)
print("Dataset root:", dataset_root)
print("Master images folder:", master_images_folder)
print("Run folder:", run_paths["run_root"])
print("Raw results CSV:", raw_results_csv)
print("Max output tokens:", MAX_OUTPUT_TOKENS)
print("Model dtype:", MODEL_DTYPE)
print("Endpoint:", BASE_URL)


In [ ]:
# ==========================================
# 5. HUGGING FACE ACCESS FOR GATED MODELS
# ==========================================
# MedGemma requires accepting the model terms on Hugging Face and setting HF_TOKEN.
from getpass import getpass
from pathlib import Path

hf_token_file = Path.home() / ".hf_token"
hf_token = medical_runtime.get_secret("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")

if not hf_token and hf_token_file.exists():
    hf_token = hf_token_file.read_text(encoding="utf-8").strip()

if not hf_token:
    hf_token = getpass("Enter your Hugging Face token for gated models: ").strip()
    if hf_token:
        hf_token_file.write_text(hf_token + "\n", encoding="utf-8")
        try:
            hf_token_file.chmod(0o600)
        except OSError:
            pass
        print("HF token saved to ~/.hf_token.")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF token loaded for this notebook session; value hidden.")
else:
    print("No HF token loaded. This is OK for ungated models, but MedGemma needs it.")


In [ ]:
# ==========================================
# 6. START OR CONNECT TO LOCAL OPENAI-COMPATIBLE SERVER
# ==========================================
previous_server_process = globals().get("server_process")
if START_SERVER and previous_server_process is not None and previous_server_process.poll() is None:
    print("Stopping previous server process before restart...")
    medical_runtime.stop_model_server(previous_server_process)

server_process = None
server_log_path = str(RUNTIME_ROOT / f"{SELECTED_MODEL_NAME}_{SERVER_ENGINE}_server.log")

if START_SERVER:
    print(f"Starting {SERVER_ENGINE} server for {SELECTED_MODEL_NAME}...")
    server_process = medical_runtime.start_model_server(
        model_name=SELECTED_MODEL_NAME,
        engine=SERVER_ENGINE,
        host=HOST,
        port=PORT,
        hf_token=hf_token,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        max_model_len=MAX_MODEL_LEN,
        extra_args=EXTRA_SERVER_ARGS,
        log_path=server_log_path,
    )
    print("Server process started. Log path:", server_log_path)
    print("Model download/load can take several minutes on the first run.")
else:
    print("START_SERVER=False; connecting to existing endpoint.")

try:
    models_payload = medical_runtime.wait_for_openai_server(
        base_url=BASE_URL,
        timeout_seconds=SERVER_TIMEOUT_SECONDS,
        process=server_process if START_SERVER else None,
        log_path=server_log_path,
    )
except Exception:
    print("Server did not become ready. Last server log lines:")
    print(medical_runtime.read_log_tail(server_log_path, lines=120))
    raise

print("Endpoint /models response:")
print(models_payload)

endpoint_client = medical_runtime.make_openai_client(BASE_URL)


In [ ]:
# ==========================================
# 7. RUN ONE-MODEL MEDICAL RADLE BENCHMARK
# ==========================================
from IPython.display import display

run_mode = "full" if TEST_LIMIT is None else f"test_limit={TEST_LIMIT}"
print(f"Run mode: {run_mode}; expected cases: {expected_case_count}; resume={RESUME}")

df_final = medical_runtime.run_medical_model_benchmark(
    client=endpoint_client,
    model_name=SELECTED_MODEL_NAME,
    image_folder=master_images_folder,
    output_csv=raw_results_csv,
    test_limit=TEST_LIMIT,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    backup_dir=raw_backup_dir,
    resume=RESUME,
)

rows = medical_runtime.validate_output_csv(raw_results_csv, SELECTED_MODEL_NAME)
print(f"Validated output CSV rows: {rows}")
if rows != expected_case_count:
    raise RuntimeError(
        f"Validated {rows} rows, expected {expected_case_count}. "
        "Do not promote or sync until the run completes cleanly."
    )
display(df_final.head())


In [ ]:
# ==========================================
# 8. SCORER VIEW + READ-ONLY AUDIT
# ==========================================
df_scorer, display_df, scorer_csv = radle_benchmark.create_scorer_view(
    raw_results_csv,
    scorer_csv=scorer_csv,
)
print("Scorer CSV:", scorer_csv)
if len(display_df.columns) > 30:
    print(f"Scorer view has {len(display_df.columns)} cases; showing first 30 columns only.")
    display(display_df.iloc[:, :30])
else:
    display(display_df)

audit_results = radle_benchmark.audit_benchmark_output(
    raw_csv=raw_results_csv,
    models=active_model,
    expected_case_ids=expected_case_ids,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

print("DATASET INTEGRITY:")
display(audit_results["dataset_integrity"])
print("BUCKET SUMMARY:")
display(audit_results["bucket_summary"])
print("STATUS SUMMARY:")
display(audit_results["status_summary"])
print("ABSTENTIONS:")
display(audit_results["abstention_summary"])
print("ANALYSIS FLAGS:")
display(audit_results["analysis_flags"].head(50))
print("REPAIR TARGETS:")
display(audit_results["repair_targets"].head(50))


if globals().get("ENABLE_VERTEX_EXPERIMENTS", False):
    try:
        import google.cloud.aiplatform as aiplatform
        aiplatform.start_run(run=run_paths["run_id"])

        # Extract simple scalar metrics to log.
        metrics = {
            "dataset_integrity_score": len(audit_results["dataset_integrity"]) if isinstance(audit_results["dataset_integrity"], list) else 1.0,
            "pending_repairs": len(audit_results["repair_targets"]),
            "total_cases": len(display_df),
        }
        aiplatform.log_metrics(metrics)
        print("Metrics logged to Vertex AI Experiments.")
        aiplatform.end_run()
    except Exception as e:
        print(f"Skipping Vertex AI logging: {e}")
else:
    print("Vertex AI logging disabled for this run.")


In [ ]:
# ==========================================
# 9. TARGETED REPAIR PLAN / RUN
# ==========================================
REPAIR_CONFIRMATION = "NO"

repair_results = radle_benchmark.run_targeted_repair(
    client=endpoint_client,
    openai_client=None,
    anthropic_client=None,
    gemini_client=None,
    image_folder=master_images_folder,
    input_csv=raw_results_csv,
    output_csv=repair_output_csv,
    repair_call_log_csv=repair_call_log_csv,
    repair_plan_csv=repair_plan_csv,
    confirmation=REPAIR_CONFIRMATION,
    models=active_model,
    backup_dir=repair_backup_dir,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

print("\nNO-API CLEANUP PLAN PREVIEW:")
display(repair_results["no_paid_cleanup_plan"].head(100))

print("\nREPAIR PLAN PREVIEW:")
display(repair_results["repair_plan"].head(100))
print("API calls this run:", repair_results["api_calls_this_run"])
print("No-API cleanups applied:", repair_results["no_paid_cleanups_applied"])
print("Repair input CSV:", raw_results_csv)
print("Repair output CSV:", repair_results["output_csv"])
print("Repair call log CSV:", repair_results["repair_call_log_csv"])


In [ ]:
# ==========================================
# 10. PROMOTE PRIVATE FINAL FILE
# ==========================================
from pathlib import Path

repair_confirmed = globals().get("REPAIR_CONFIRMATION", "NO") != "NO"
repair_output_ready = repair_confirmed and Path(repair_output_csv).exists()
private_final_source_csv = repair_output_csv if repair_output_ready else raw_results_csv
private_final_source_label = "repaired" if repair_output_ready else "raw"

ALLOW_PROMOTE_WITH_PENDING_REPAIRS = False
ALLOW_PROMOTE_WITH_INTEGRITY_WARNINGS = False

# Guardrail: check expected rows/cases and audit targets before promotion.
audit_pre_promote = radle_benchmark.audit_benchmark_output(
    raw_csv=private_final_source_csv,
    models=active_model,
    expected_case_ids=expected_case_ids,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)
pending_repairs = len(audit_pre_promote["repair_targets"])
integrity_table = audit_pre_promote["dataset_integrity"]
integrity = dict(zip(integrity_table["metric"], integrity_table["value"]))


def _as_int(value):
    try:
        return int(value)
    except Exception:
        return -1


integrity_problems = []
if _as_int(integrity.get("rows")) != expected_case_count:
    integrity_problems.append(f"rows={integrity.get('rows')} expected={expected_case_count}")
if _as_int(integrity.get("unique_cases")) != expected_case_count:
    integrity_problems.append(f"unique_cases={integrity.get('unique_cases')} expected={expected_case_count}")
for metric in ["duplicate_case_ids", "missing_expected_case_ids", "extra_case_ids"]:
    if str(integrity.get(metric, "none")) != "none":
        integrity_problems.append(f"{metric}={integrity.get(metric)}")

if integrity_problems and not ALLOW_PROMOTE_WITH_INTEGRITY_WARNINGS:
    print("\nWARNING: dataset integrity problems remain. Promotion halted.")
    for problem in integrity_problems:
        print("-", problem)
    print("Fix the run, or manually set ALLOW_PROMOTE_WITH_INTEGRITY_WARNINGS=True to override.")
elif pending_repairs > 0 and not ALLOW_PROMOTE_WITH_PENDING_REPAIRS:
    print(f"\nWARNING: {pending_repairs} case-model cells still need repair. Promotion halted.")
    print("Run a full repair, or manually set ALLOW_PROMOTE_WITH_PENDING_REPAIRS=True to override.")
else:
    final_manifest = radle_benchmark.promote_final_results(
        source_csv=private_final_source_csv,
        final_csv=final_results_csv,
        manifest_json=final_manifest_json,
        run_id=run_paths["run_id"],
        source_label=private_final_source_label,
        metadata={
            "run_label": RUN_LABEL,
            "test_limit": TEST_LIMIT if TEST_LIMIT is not None else "full",
            "expected_case_count": expected_case_count,
            "max_output_tokens": MAX_OUTPUT_TOKENS,
        },
    )

    print("Private final source:", private_final_source_csv)
    print("Private final CSV:", final_results_csv)
    print("Private final manifest:", final_manifest_json)
    print("Private final SHA256:", final_manifest["sha256"])


In [ ]:
# ==========================================
# 11. EXPORT ANSWER-FREE PUBLIC RELEASE TABLES
# ==========================================
from pathlib import Path

if not Path(final_results_csv).exists():
    print("ERROR: final_results_csv does not exist. Promotion was likely halted.")
else:
    public_results_source_csv = final_results_csv
    public_call_log_csv = repair_call_log_csv if Path(repair_call_log_csv).exists() else None

    public_release_files = radle_benchmark.export_public_release_tables(
        results_csv=public_results_source_csv,
        output_dir=public_release_dir,
        models=active_model,
        call_log_csv=public_call_log_csv,
        run_id=run_paths["run_id"],
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )

    print("Public release source CSV:", public_results_source_csv)
    print("Public case-model CSV:", public_release_files["case_model_csv"])
    print("Public model summary CSV:", public_release_files["summary_csv"])
    print("Public sanitized call log CSV:", public_release_files["sanitized_call_log_csv"])
    print("Public manifest:", public_release_files["manifest_json"])


In [ ]:
# ==========================================
# 12. SYNC RESULTS TO GCS
# ==========================================
from pathlib import Path

RESULTS_GCS_URI = os.environ.get("RADLE_RESULTS_GCS_URI", "").rstrip("/")
if not RESULTS_GCS_URI:
    raise RuntimeError("Set RADLE_RESULTS_GCS_URI to the private GCS runs folder.")

local_run_root = Path(run_paths["run_root"])
remote_run_root = f"{RESULTS_GCS_URI}/{run_paths['run_id']}"

if not local_run_root.exists():
    raise RuntimeError(f"Local run folder does not exist: {local_run_root}")

run_command([
    "gcloud",
    "storage",
    "rsync",
    "-r",
    str(local_run_root),
    remote_run_root,
])

print("Synced local run folder:", local_run_root)
print("Synced GCS run folder:", remote_run_root)


In [ ]:
# ==========================================
# 13. STOP LOCAL SERVER WHEN DONE
# ==========================================
# Run this cell before switching SELECTED_MODEL_NAME, or restart the runtime.
medical_runtime.stop_model_server(server_process)
print("Server stopped if it was started by this notebook.")
